# Enrollee-Bot experiments

RAG pipeline for FCS HSE admission chatbot. 
Experiments: preprocessing, chunking, embedders, reranker, generation.

In [ ]:
# !pip install -q sentence-transformers faiss-cpu langchain langchain-experimental \
#                 openai ragas datasets pandas numpy scikit-learn tqdm pymorphy3

In [ ]:
import json, re, os, time, unicodedata, random
from pathlib import Path
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

DATA_DIR = Path("./data")

## Data

In [ ]:
raw_corpus = (DATA_DIR / "text.txt").read_text(encoding="utf-8")
qa_dataset = json.loads((DATA_DIR / "fcs_hse_qa_dataset.json").read_text(encoding="utf-8"))

print(len(raw_corpus), "chars in corpus")
print(len(qa_dataset), "qa pairs")
pd.DataFrame(qa_dataset)['category'].value_counts()

## Preprocessing

In [ ]:
def clean_ws(text):
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    text = re.sub(r'(?<=[а-яёa-z,])\n(?=[а-яёa-z])', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def strip_headers(text):
    patterns = [
        r'^\s*НИУ ВШЭ\s*$',
        r'^\s*стр\.?\s*\d+\s*(из\s*\d+)?\s*$',
        r'^\s*\d+\s*/\s*\d+\s*$',
        r'^\s*\d+\s*$',
        r'^\s*Факультет компьютерных наук\s*$',
    ]
    for p in patterns:
        text = re.sub(p, '', text, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

def norm_unicode(text):
    text = unicodedata.normalize('NFKC', text)
    for a, b in {'«':'"', '»':'"', '\u201c':'"', '\u201d':'"',
                  '\u2014':'-', '\u2013':'-', '\u2212':'-',
                  '\xa0':' ', '\u2009':' '}.items():
        text = text.replace(a, b)
    return text

def apply_tables(text):
    p = DATA_DIR / "tables_clean.json"
    if not p.exists():
        return text
    for r in json.loads(p.read_text(encoding="utf-8")):
        text = text.replace(r['old'], r['new'])
    return text

preprocess = {
    "baseline":     lambda t: t,
    "+ ws":         clean_ws,
    "+ headers":    lambda t: strip_headers(clean_ws(t)),
    "+ unicode":    lambda t: norm_unicode(strip_headers(clean_ws(t))),
    "+ tables":     lambda t: apply_tables(norm_unicode(strip_headers(clean_ws(t)))),
}

for k, fn in preprocess.items():
    print(f"{k:12s} {len(fn(raw_corpus))}")

## Chunking, index, metrics

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

def chunk_fixed(text, size=500, overlap=0.1):
    s = CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=int(size*overlap))
    return s.split_text(text)

def chunk_recursive(text, size=500, overlap=0.1):
    s = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=int(size*overlap),
        separators=["\n\n", "\n", ". ", " ", ""])
    return s.split_text(text)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

class Index:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)
        self.chunks = []
        self.idx = None

    def build(self, chunks):
        self.chunks = chunks
        emb = self.model.encode(chunks, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.idx = faiss.IndexFlatIP(emb.shape[1])
        self.idx.add(emb.astype('float32'))

    def search(self, q, k=5):
        qv = self.model.encode([q], normalize_embeddings=True)
        s, i = self.idx.search(qv.astype('float32'), k)
        return list(zip(i[0].tolist(), s[0].tolist()))

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

def lemmas(text):
    return {morph.parse(t)[0].normal_form for t in re.findall(r'\w+', text.lower()) if len(t) > 2}

def is_hit(chunk, ref, min_match=2):
    return len(lemmas(chunk) & lemmas(ref)) >= min_match

def hit_rate(index, qa, k=5):
    return sum(any(is_hit(index.chunks[i], it['answer']) for i, _ in index.search(it['question'], k))
               for it in qa) / len(qa)

def mrr(index, qa, k=10):
    s = 0.0
    for it in qa:
        for rank, (i, _) in enumerate(index.search(it['question'], k), start=1):
            if is_hit(index.chunks[i], it['answer']):
                s += 1.0 / rank
                break
    return s / len(qa)

## Exp 1: preprocessing

In [ ]:
EMB = "intfloat/multilingual-e5-small"

rows = []
for name, fn in preprocess.items():
    chunks = chunk_fixed(fn(raw_corpus), 500, 0.1)
    idx = Index(EMB); idx.build(chunks)
    rows.append({"config": name, "HR@5": hit_rate(idx, qa_dataset), "chunks": len(chunks)})

pd.DataFrame(rows)

## Exp 2: chunking

In [ ]:
clean = preprocess['+ tables'](raw_corpus)

configs = [
    ("fixed 256/10",   lambda t: chunk_fixed(t, 256, 0.1)),
    ("fixed 500/10",   lambda t: chunk_fixed(t, 500, 0.1)),
    ("fixed 500/20",   lambda t: chunk_fixed(t, 500, 0.2)),
    ("fixed 800/10",   lambda t: chunk_fixed(t, 800, 0.1)),
    ("fixed 1200/10",  lambda t: chunk_fixed(t, 1200, 0.1)),
    ("recursive 500",  lambda t: chunk_recursive(t, 500, 0.1)),
    ("recursive 800",  lambda t: chunk_recursive(t, 800, 0.1)),
]

rows = []
for name, fn in configs:
    chunks = fn(clean)
    idx = Index(EMB); idx.build(chunks)
    rows.append({
        "method": name,
        "HR@5": hit_rate(idx, qa_dataset),
        "MRR@10": mrr(idx, qa_dataset),
        "avg_tok": int(np.mean([len(c.split()) for c in chunks]) * 1.4),
    })

pd.DataFrame(rows)

## Exp 3: embedders

In [ ]:
models = [
    "cointegrated/rubert-tiny2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "BAAI/bge-m3",
]

chunks_prod = chunk_recursive(clean, 800, 0.1)

rows = []
for m in models:
    t = time.time()
    idx = Index(m); idx.build(chunks_prod)
    rows.append({
        "model": m.split("/")[-1],
        "HR@5": hit_rate(idx, qa_dataset),
        "MRR@10": mrr(idx, qa_dataset),
        "build_s": round(time.time() - t, 1),
    })

pd.DataFrame(rows)

## Exp 4: reranker

In [ ]:
from sentence_transformers import CrossEncoder

base = Index("intfloat/multilingual-e5-base"); base.build(chunks_prod)

rerankers = {
    "none":           None,
    "ms-marco-L12":   "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "DiTy ru":        "DiTy/cross-encoder-russian-msmarco",
    "bge-v2-m3":      "BAAI/bge-reranker-v2-m3",
}

def eval_rerank(model_name, n=20, k=5):
    ce = CrossEncoder(model_name) if model_name else None
    hits, mrr_s, lat = 0, 0.0, []
    for it in qa_dataset:
        t = time.time()
        cands = [(i, base.chunks[i]) for i, _ in base.search(it['question'], k=n)]
        if ce:
            scores = ce.predict([[it['question'], c] for _, c in cands])
            cands = [cands[i] for i in np.argsort(-scores)]
        cands = cands[:k]
        lat.append(time.time() - t)
        if any(is_hit(c, it['answer']) for _, c in cands):
            hits += 1
        for rank, (_, c) in enumerate(cands, start=1):
            if is_hit(c, it['answer']):
                mrr_s += 1.0 / rank
                break
    return hits/len(qa_dataset), mrr_s/len(qa_dataset), np.mean(lat)

rows = []
for name, m in rerankers.items():
    hr, mr, l = eval_rerank(m)
    rows.append({"reranker": name, "HR@5": hr, "MRR@10": mr, "lat_s": round(l, 2)})

pd.DataFrame(rows)

## Exp 5: generation

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def retrieve(q, n=20, k=5):
    cands = [base.chunks[i] for i, _ in base.search(q, k=n)]
    scores = reranker.predict([[q, c] for c in cands])
    return [cands[i] for i in np.argsort(-scores)[:k]]

SYS = ("Ты - ассистент приёмной комиссии ФКН ВШЭ. "
       "Отвечай только по контексту, без внешних знаний. "
       "Если ответа нет - предложи priem-fcs@hse.ru. "
       "В конце добавляй ссылку на источник.")

FEWSHOT = """
Q: Когда подаются документы?
Ctx: Приём с 20.06 по 25.07.2026.
A: Документы принимаются с 20 июня по 25 июля 2026 г. Источник: правила приёма 2026.

Q: Какая погода завтра?
Ctx: [нет]
A: Отвечаю только по вопросам поступления на ФКН.

Q: БВИ за олимпиаду X?
Ctx: [нет упоминания]
A: В базе нет данных по этой олимпиаде. Напишите на priem-fcs@hse.ru.
"""

def answer(q, ctx, model="gpt-4o-mini", fewshot=False):
    sys = SYS + ("\n\n" + FEWSHOT if fewshot else "")
    msg = [{"role": "system", "content": sys},
           {"role": "user", "content": f"Контекст:\n{chr(10).join(ctx)}\n\nВопрос: {q}"}]
    r = client.chat.completions.create(model=model, messages=msg, temperature=0.1, max_tokens=500)
    return r.choices[0].message.content

In [ ]:
OOD = [
    "Напиши эссе про лето",
    "Реши уравнение x^2 - 5x + 6 = 0",
    "Что думаешь о политике?",
    "Какая погода завтра?",
    "Какие у меня шансы поступить с 280 баллами?",
]

REFUSAL_MARKERS = ["к сожалению", "не могу", "только", "не отвечаю", "приёмную комиссию", "в базе нет"]

def is_refusal(a):
    a = a.lower()
    return any(m in a for m in REFUSAL_MARKERS)

def eval_gen(model, fewshot):
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
    from datasets import Dataset

    rows = []
    for it in qa_dataset:
        c = retrieve(it['question'])
        rows.append({
            "question": it['question'],
            "answer": answer(it['question'], c, model, fewshot),
            "contexts": c,
            "ground_truth": it['answer'],
        })

    scores = evaluate(Dataset.from_list(rows),
                       metrics=[faithfulness, answer_relevancy, answer_correctness])

    refusals = sum(is_refusal(answer(q, retrieve(q), model, fewshot)) for q in OOD)

    return {
        "model": model, "fewshot": fewshot,
        "faith": float(scores['faithfulness']),
        "rel": float(scores['answer_relevancy']),
        "corr": float(scores['answer_correctness']),
        "refusal": refusals / len(OOD),
    }

In [ ]:
results = [
    eval_gen("gpt-4o-mini", False),
    eval_gen("gpt-4o-mini", True),
    eval_gen("gpt-4o", True),
]
pd.DataFrame(results)

## Final config

- Preprocessing: 5 steps (ws + headers + unicode + tables)
- Chunking: recursive 800/10%
- Embedder: multilingual-e5-base
- Reranker: bge-reranker-v2-m3 (top-20 → top-5)
- Generator: gpt-4o-mini + few-shot
- Fallback LLM: YandexGPT-4 Pro